In [1]:
# ETL MIMIC-III complet
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("HospitalOptimizer-ETL-MIMIC") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark {spark.version} démarré")

✅ Spark 3.5.0 démarré


In [2]:
BASE_PATH = "/home/jovyan/data/raw/mimic_synthetic"

patients      = spark.read.csv(f"{BASE_PATH}/PATIENTS.csv",      header=True, inferSchema=True)
admissions    = spark.read.csv(f"{BASE_PATH}/ADMISSIONS.csv",    header=True, inferSchema=True)
icustays      = spark.read.csv(f"{BASE_PATH}/ICUSTAYS.csv",      header=True, inferSchema=True)
labevents     = spark.read.csv(f"{BASE_PATH}/LABEVENTS.csv",     header=True, inferSchema=True)
prescriptions = spark.read.csv(f"{BASE_PATH}/PRESCRIPTIONS.csv", header=True, inferSchema=True)

print("✅ Tables chargées :")
print(f"   PATIENTS      : {patients.count():>10,} lignes")
print(f"   ADMISSIONS    : {admissions.count():>10,} lignes")
print(f"   ICUSTAYS      : {icustays.count():>10,} lignes")
print(f"   LABEVENTS     : {labevents.count():>10,} lignes")
print(f"   PRESCRIPTIONS : {prescriptions.count():>10,} lignes")

✅ Tables chargées :
   PATIENTS      :     50,000 lignes
   ADMISSIONS    :     84,887 lignes
   ICUSTAYS      :     33,955 lignes
   LABEVENTS     : 14,601,058 lignes
   PRESCRIPTIONS :  2,190,401 lignes


In [3]:
# Jointure des tables
# ── Étape 1 : ADMISSIONS + PATIENTS ──────────────────────────────
adm_patients = admissions.join(
    patients.select("subject_id", "gender", "dob", "expire_flag"),
    on="subject_id",
    how="left"
)

# ── Étape 2 : Ajouter ICUSTAYS ────────────────────────────────────
adm_icu = adm_patients.join(
    icustays.select("hadm_id", "icustay_id", "first_careunit", "los"),
    on="hadm_id",
    how="left"
)

# ── Étape 3 : Agréger LABEVENTS par séjour ────────────────────────
labs_agg = labevents.groupBy("hadm_id").agg(
    F.count("row_id").alias("num_labs"),
    F.countDistinct("itemid").alias("num_distinct_labs"),
    F.sum(F.when(F.col("flag") == "abnormal", 1).otherwise(0)).alias("num_abnormal_labs")
)

adm_labs = adm_icu.join(labs_agg, on="hadm_id", how="left")

# ── Étape 4 : Agréger PRESCRIPTIONS par séjour ───────────────────
rx_agg = prescriptions.groupBy("hadm_id").agg(
    F.count("row_id").alias("num_rx"),
    F.countDistinct("drug").alias("num_distinct_drugs")
)

df_joined = adm_labs.join(rx_agg, on="hadm_id", how="left")

print(f"✅ Jointure complète")
print(f"   Lignes   : {df_joined.count():,}")
print(f"   Colonnes : {len(df_joined.columns)}")
df_joined.show(3)

✅ Jointure complète
   Lignes   : 84,887
   Colonnes : 27
+-------+----------+-------------------+-------------------+---------+--------------+--------------------+--------------------+---------+--------+--------+--------------+---------------+--------------------+--------------------+--------------------+------+----------+-----------+----------+--------------+----+--------+-----------------+-----------------+------+------------------+
|hadm_id|subject_id|          admittime|          dischtime|deathtime|admission_type|  admission_location|  discharge_location|insurance|language|religion|marital_status|      ethnicity|           diagnosis|hospital_expire_flag|has_chartevents_data|gender|       dob|expire_flag|icustay_id|first_careunit| los|num_labs|num_distinct_labs|num_abnormal_labs|num_rx|num_distinct_drugs|
+-------+----------+-------------------+-------------------+---------+--------------+--------------------+--------------------+---------+--------+--------+--------------+--------

In [6]:
# Calcul de LOSdays et sauvegarde Bronze
# ── Calculer LOSdays depuis admittime et dischtime ────────────────
df_joined = df_joined.withColumn(
    "LOSdays",
    F.round(
        (F.unix_timestamp("dischtime") - F.unix_timestamp("admittime")) / 86400, 2
    )
)

# ── Sauvegarder en Bronze ─────────────────────────────────────────
df_joined.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/bronze/mimic_full_raw.parquet")

print("✅ Bronze sauvegardé")
print(f"   Lignes   : {df_joined.count():,}")
print(f"   Colonnes : {len(df_joined.columns)}")
print(f"\nAperçu LOSdays :")
df_joined.select("hadm_id", "admittime", "dischtime", "LOSdays").show(5)

✅ Bronze sauvegardé
   Lignes   : 84,887
   Colonnes : 28

Aperçu LOSdays :
+-------+-------------------+-------------------+-------+
|hadm_id|          admittime|          dischtime|LOSdays|
+-------+-------------------+-------------------+-------+
| 100000|2005-01-02 14:33:10|2005-01-03 11:30:01|   0.87|
| 100001|2009-01-31 00:21:38|2009-02-08 14:30:32|   8.59|
| 100002|2007-03-01 09:54:12|2007-03-05 18:36:58|   4.36|
| 100003|2008-04-01 16:33:21|2008-04-03 03:54:43|   1.47|
| 100004|2002-06-29 03:35:55|2002-07-11 21:00:05|  12.73|
+-------+-------------------+-------------------+-------+
only showing top 5 rows



In [7]:
# Couche Silver — Nettoyage
# ── Lire depuis Bronze ────────────────────────────────────────────
df_silver = spark.read.parquet("/home/jovyan/data/bronze/mimic_full_raw.parquet")

# ── Imputer les valeurs manquantes ────────────────────────────────
df_silver = df_silver \
    .fillna("UNKNOWN", subset=["marital_status", "religion", "language"]) \
    .fillna(0, subset=["num_labs", "num_distinct_labs",
                       "num_abnormal_labs", "num_rx",
                       "num_distinct_drugs", "los"])

# ── Filtrer les outliers ──────────────────────────────────────────
df_silver = df_silver.filter(
    (df_silver.LOSdays > 0) &
    (df_silver.LOSdays <= 200)
)

# ── Supprimer colonnes inutiles ───────────────────────────────────
df_silver = df_silver.drop("expire_flag", "dob")

# ── Sauvegarder ──────────────────────────────────────────────────
df_silver.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/silver/mimic_full_clean.parquet")

print("✅ Silver sauvegardé")
print(f"   Lignes   : {df_silver.count():,}")
print(f"   Colonnes : {len(df_silver.columns)}")

✅ Silver sauvegardé
   Lignes   : 84,886
   Colonnes : 26


In [8]:
# Couche Gold — Features métier
from pyspark.sql.window import Window

# ── Lire depuis Silver ────────────────────────────────────────────
df_gold = spark.read.parquet("/home/jovyan/data/silver/mimic_full_clean.parquet")

# ── Feature 1 : score de complexité ──────────────────────────────
df_gold = df_gold.withColumn(
    "complexity_score",
    F.round((df_gold.num_labs + df_gold.num_rx + df_gold.num_distinct_drugs) / 3, 2)
)

# ── Feature 2 : taux de labs anormaux ────────────────────────────
df_gold = df_gold.withColumn(
    "abnormal_lab_rate",
    F.round(
        F.when(df_gold.num_labs > 0,
            df_gold.num_abnormal_labs / df_gold.num_labs
        ).otherwise(0), 3
    )
)

# ── Feature 3 : patient à risque ─────────────────────────────────
df_gold = df_gold.withColumn(
    "high_risk",
    F.when(df_gold.LOSdays > 6.5, 1).otherwise(0)
)

# ── Feature 4 : durée moyenne par type d'admission ───────────────
window_admit = Window.partitionBy("admission_type")
df_gold = df_gold.withColumn(
    "avg_los_by_admit_type",
    F.round(F.avg("LOSdays").over(window_admit), 2)
)

# ── Feature 5 : durée moyenne par service ICU ────────────────────
window_icu = Window.partitionBy("first_careunit")
df_gold = df_gold.withColumn(
    "avg_los_by_careunit",
    F.round(F.avg("LOSdays").over(window_icu), 2)
)

# ── Sauvegarder ──────────────────────────────────────────────────
df_gold.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/gold/mimic_full_features.parquet")

print("✅ Gold sauvegardé")
print(f"   Lignes   : {df_gold.count():,}")
print(f"   Colonnes : {len(df_gold.columns)}")
print("\nAperçu des nouvelles features :")
df_gold.select(
    "hadm_id", "LOSdays", "complexity_score",
    "abnormal_lab_rate", "high_risk",
    "avg_los_by_admit_type"
).show(5)

✅ Gold sauvegardé
   Lignes   : 84,886
   Colonnes : 31

Aperçu des nouvelles features :
+-------+-------+----------------+-----------------+---------+---------------------+
|hadm_id|LOSdays|complexity_score|abnormal_lab_rate|high_risk|avg_los_by_admit_type|
+-------+-------+----------------+-----------------+---------+---------------------+
| 102119|  10.87|            85.0|            0.302|        1|                  9.1|
| 102594|   3.14|           27.33|            0.203|        0|                  9.1|
| 102793|   1.13|            8.33|            0.353|        0|                  9.1|
| 103357|   5.21|           43.33|            0.276|        0|                  9.1|
| 103747|  10.06|           91.67|            0.221|        1|                  9.1|
+-------+-------+----------------+-----------------+---------+---------------------+
only showing top 5 rows

